### <b>Phase 1: Data Collection</b>

<b>Task 1.1 - Books to Scrape (HTML, Requests, BeautifulSoup)</b>

- Retrieve the first three catalogue pages (approximately 60 books total when all pages are accessible).
- Use Requests with a timeout and status checks; parse the HTML using BeautifulSoup.
- Extract title, displayed GBP price, star rating, availability status, absolute product URL, and page number.
- Demonstrate pagination using a loop, and handle missing tags/attributes without crashing.
- Export the raw results to data/raw/books_raw.csv and preserve the extraction timestamp

In [ ]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from pathlib import Path
from datetime import datetime, timezone
import json
import pandas as pd

rating_map = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}

books_raw = []
failed_pages = []
skipped_books = []

extraction_time = datetime.now(timezone.utc).isoformat()

for page in range(1, 4):
    url = f"https://books.toscrape.com/catalogue/page-{page}.html"
    try:
        r = requests.get(url, timeout=15)
        r.raise_for_status()
        r.encoding = r.apparent_encoding
        soup = BeautifulSoup(r.text, "html.parser")
        books = soup.select(".product_pod")
        
        if not books:
            print(f"Warning: no books found on page {page}")
            failed_pages.append(page)
            continue

        for position, book in enumerate(books, start=1):
            title_tag = book.select_one("h3 a")
            price_tag = book.select_one("p.price_color")
            rating_tag = book.select_one("p.star-rating")
            availability_tag = book.select_one("p.instock.availability")

            title = title_tag.get("title") if title_tag else None
            price = price_tag.get_text(strip=True) if price_tag else None
            rating_classes = rating_tag.get("class", []) if rating_tag else []  # ["star-rating", "Three"] 
            
            rating = next((value for value in rating_classes if value in rating_map), None)
            
            availability = availability_tag.get_text(strip=True) if availability_tag else None
            relative_url = title_tag.get("href") if title_tag else None
            absolute_url = urljoin(url, relative_url) if relative_url else None

            if any(value is None or value == "" for value in (title, price, rating, availability, absolute_url)):
                print(f"Warning: missing book field on page {page}, item {position}; saved as missing")

            books_raw.append({
                "Title": title,
                "Price": price,
                "Rating": rating,
                "availability": availability,
                "AbsoluteUrl": absolute_url,
                "Page": page
            })
    except requests.RequestException as e:
        failed_pages.append(page)
        print(f"Request error on page {page}: {e}")

raw_path = Path("../data/raw")
raw_path.mkdir(parents=True, exist_ok=True)
pd.DataFrame(books_raw).to_csv(raw_path / "books_raw.csv", index=False)


# Transformations are applied to a separate DataFrame, never to the raw extract.
df = pd.DataFrame(books_raw)
if not df.empty:
    df["Price"] = pd.to_numeric(
        df["Price"].str.replace("Â", "", regex=False).str.replace("£", "", regex=False),
        errors="coerce"
    )
    df["Rating"] = df["Rating"].map(rating_map)
    bad_prices = int(df["Price"].isna().sum())
    if bad_prices:
        print(f"Warning: {bad_prices} book prices could not be converted")
else:
    print("Warning: no book records collected")

clean_path = Path("../data/clean")
clean_path.mkdir(parents=True, exist_ok=True)
df.to_csv(clean_path / "books_clean.csv", index=False)

with open(raw_path / "books_extraction_metadata.json", "w", encoding="utf-8") as file:
    json.dump({
        "source": "https://books.toscrape.com/",
        "extracted_at_utc": extraction_time,
        "requested_pages": [1, 2, 3],
        "failed_or_empty_pages": failed_pages,
        "record_count": len(books_raw)
    }, file, indent=4)

print(f"Books collected: {len(books_raw)}; failed/empty pages: {failed_pages}")


Books collected: 60; failed/empty pages: []


<b>Task 1.2 - Fake Store API (REST, JSON)</b>

- Send an HTTP GET request to https://fakestoreapi.com/products.
- Check HTTP status and parse JSON; verify the top-level structure and number of records.
- Select ID, title, price, category, and rating fields (including nested rating information).
- Export a raw JSON copy to data/raw/products_raw.json and a CSV snapshot if desired

In [11]:
import json
import pandas as pd
from pathlib import Path
from datetime import datetime, timezone

# The API was unavailable from Python; its original response was downloaded in a browser.
# Preserve products_raw.json unchanged as the source snapshot.

raw_path = Path("../data/raw")

with open(raw_path / "products_raw.json", "r", encoding="utf-8-sig") as f:
    products = json.load(f)

if not isinstance(products, list):
    raise ValueError("Expected a JSON list of products")

print("Product records:", len(products))

processed_at = datetime.now(timezone.utc).isoformat()

with open(raw_path / "products_source_metadata.json", "w", encoding="utf-8") as file:
    json.dump({
        "source": "https://fakestoreapi.com/products",
        "method": "Manual browser download because the Python request failed",
        "original_download_time_utc": None,
        "processed_at_utc": processed_at,
        "record_count": len(products)
    }, file, indent=4)

df = pd.json_normalize(products)
df.to_csv(raw_path / "products_raw.csv", index=False)

required_product_cols = ["id", "title", "price", "category", "rating.rate", "rating.count"]
missing = [col for col in required_product_cols if col not in df.columns]
if missing:
    raise ValueError(f"Missing required product fields: {missing}")

df = df[required_product_cols].rename(columns={
    "rating.rate": "rating_rate",
    "rating.count": "rating_count"
})

clean_path = Path("../data/clean")
clean_path.mkdir(parents=True, exist_ok=True)
with open(clean_path / "products_clean.json", "w", encoding="utf-8") as file:
    json.dump(df.to_dict(orient="records"), file, indent=4, ensure_ascii=False)
df.to_csv(clean_path / "products_clean.csv", index=False)


Product records: 20


<b>Task 1.3 - Amazon Stock History (yfinance)</b>

- Create a yfinance Ticker object using AMZN.
- Request the latest available 1-year historical series at daily intervals.
- Reset the date index safely, inspect column names, and preserve OHLCV fields.
- Save the historical extract as data/raw/amzn_raw.csv.
- Document whether automatic price adjustment was enabled and how this affects Open, High, Low, and Close.


In [ ]:
import yfinance as yf
import pandas as pd
import json
from pathlib import Path
from datetime import datetime, timezone

amzn = yf.Ticker("AMZN")
df = amzn.history(period="1y", interval="1d", auto_adjust=False)
extraction_time = datetime.now(timezone.utc).isoformat()

# auto_adjust=False disables yfinance's additional adjustment of OHLC prices.
# Adj Close remains separate when supplied; source values may already account for splits.

if df.empty:
    print("No AMZN data returned; existing files were not overwritten")
else:
    df = df.reset_index()
    required_cols = ["Date", "Open", "High", "Low", "Close", "Volume"]
    missing = [col for col in required_cols if col not in df.columns]
    if missing:
        print(f"Missing required AMZN columns: {missing}; files were not overwritten")
    else:
        raw_path = Path("../data/raw")
        clean_path = Path("../data/clean")
        raw_path.mkdir(parents=True, exist_ok=True)
        clean_path.mkdir(parents=True, exist_ok=True)
        
        df.to_csv(raw_path / "amzn_raw.csv", index=False)
        
        df[required_cols].to_csv(clean_path / "amzn_clean.csv", index=False)
        
        with open(raw_path / "amzn_extraction_metadata.json", "w", encoding="utf-8") as file:
            json.dump({
                "ticker": "AMZN",
                "extracted_at_utc": extraction_time,
                "period": "1y",
                "interval": "1d",
                "auto_adjust": False,
                "record_count": len(df)
            }, file, indent=4)
        print(f"AMZN records saved: {len(df)}")


AMZN records saved: 251


d:\Projects\retail_intelligence_data_analysis\.venv\Lib\site-packages\yfinance\utils.py:568: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quotes.sort_index(inplace=True)


### <b>Phase 2 - Data Validation and Cleaning</b>
- <b>2.1 Profile:</b> Inspect row/column counts, inferred data types, sample records, missing values, and ranges.
- <b>2.2 Missing values:</b> Choose drop, fill, or retain, explaining why; do not indiscriminately delete rows.
- <b>2.3 Duplicates:</b> Find duplicates using stable identifiers (product URL / product ID / date), not only entire-row equality.
- <b>2.4 Text and numbers:</b> Convert book prices and rating strings into meaningful numeric fields; handle currency symbols and separators.
- <b>2.5 Schemas:</b> Use consistent snake_case headers and validate price, rating, date, and stock OHLCV types.
- <b>2.6 Export:</b> Write cleaned product and book CSVs; export at least one cleaned dataset to XML as a file-format exercise.
- <b>2.7 Quality log:</b> Record counts before/after cleaning, transformations, excluded rows, and unresolved concerns.
- Suggested cleaned outputs: data/clean/books_clean.csv, data/clean/products_clean.csv, data/clean/amzn_clean.csv, data/clean/products_clean.xml. Keep the original raw extracts unchanged.

In [ ]:
from pandas.api.types import is_numeric_dtype

amzn_raw = pd.read_csv("../data/raw/amzn_raw.csv")
products_raw = pd.read_csv("../data/raw/products_raw.csv")
book_raw = pd.read_csv("../data/raw/books_raw.csv")

# 2.1 Data profiling
# ----------------------------------------

# # Amazon Stocks:
# print(amzn_raw.shape)  # row/columns counts
# print(amzn_raw.dtypes)  # data types
# print(amzn_raw.head()) # sample records
# print(amzn_raw.isna().sum())   # missing values
# print(amzn_raw.describe(include="all")) # ranges


# # Products:
# print(products_raw.shape) # row/columns counts
# print(products_raw.dtypes) # data types
# print(products_raw.head()) # sample records
# print(products_raw.isna().sum()) # missing values
# print(products_raw.describe(include="all")) # ranges

# # Books:

# print(book_raw.shape) # row/columns counts
# print(book_raw.dtypes) # data types
# print(book_raw.head()) # sample records
# print(book_raw.isna().sum()) # missing values
# print(book_raw.describe(include="all")) # ranges



# 2.2 Missing Values
# ----------------------------

amzn_clean = amzn_raw.copy()
book_clean = book_raw.copy()
products_clean = products_raw.copy()

# Chech if there are missing values in the rows, I use boolean indexing

print(book_clean[book_clean.isna().any(axis=1)])

print(amzn_clean[amzn_clean.isna().any(axis=1)])

print(products_clean[products_clean.isna().any(axis=1)])

# We don't have to delete Missing values becuase they don't exist :)


# 2.2 Check duplicates
# ----------------------------

print(book_clean.duplicated(subset=["AbsoluteUrl"]).sum())
print(products_clean.duplicated(subset=["id"]).sum())
print(amzn_clean.duplicated(subset=["Date"]).sum())


# 2.3 Text and numbers in books:
# ----------------------------

book_clean["Price"] = pd.to_numeric(
        book_clean["Price"].str.replace("Â", "", regex=False).str.replace("£", "", regex=False),
        errors="coerce")

book_clean["Rating"] = book_clean["Rating"].map(rating_map)

# print(book_clean["Price"])




# 2.4 Use consistent snake_case headers and validate price, rating, date, and stock OHLCV types.
# ----------------------------






Empty DataFrame
Columns: [Title, Price, Rating, availability, AbsoluteUrl, Page]
Index: []
Empty DataFrame
Columns: [Date, Open, High, Low, Close, Adj Close, Volume, Dividends, Stock Splits]
Index: []
Empty DataFrame
Columns: [id, title, price, description, category, image, rating.rate, rating.count]
Index: []
0
0
0
